<a href="https://colab.research.google.com/github/nafaniaa/innowise-data-engineer-intership/blob/spark-task-google-colab/spark-task-google-colab/pyspark_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, desc
from pyspark.sql.functions import sum as spark_sum

In [3]:
spark = SparkSession.builder \
    .appName("Test") \
    .master("local[*]") \
    .getOrCreate()

print("Spark works!")

Spark works!


In [4]:
from google.colab import files
uploaded = files.upload()

Saving actor.csv to actor.csv
Saving address.csv to address.csv
Saving category.csv to category.csv
Saving city.csv to city.csv
Saving customer.csv to customer.csv
Saving film.csv to film.csv
Saving film_actor.csv to film_actor.csv
Saving film_category.csv to film_category.csv
Saving inventory.csv to inventory.csv
Saving payment.csv to payment.csv
Saving rental.csv to rental.csv


In [5]:
film = spark.read.csv("film.csv", header=True, inferSchema=True)
category = spark.read.csv("category.csv", header=True, inferSchema=True)
film_category = spark.read.csv("film_category.csv", header=True, inferSchema=True)
actor = spark.read.csv("actor.csv", header=True, inferSchema=True)
film_actor = spark.read.csv("film_actor.csv", header=True, inferSchema=True)
inventory = spark.read.csv("inventory.csv", header=True, inferSchema=True)
rental = spark.read.csv("rental.csv", header=True, inferSchema=True)
payment = spark.read.csv("payment.csv", header=True, inferSchema=True)
customer = spark.read.csv("customer.csv", header=True, inferSchema=True)
address = spark.read.csv("address.csv", header=True, inferSchema=True)
city = spark.read.csv("city.csv", header=True, inferSchema=True)

In [6]:
film.show(5)

+-------+----------------+--------------------+------------+-----------+--------------------+---------------+-----------+------+----------------+------+--------------------+--------------------+--------------------+
|film_id|           title|         description|release_year|language_id|original_language_id|rental_duration|rental_rate|length|replacement_cost|rating|         last_update|    special_features|            fulltext|
+-------+----------------+--------------------+------------+-----------+--------------------+---------------+-----------+------+----------------+------+--------------------+--------------------+--------------------+
|      1|ACADEMY DINOSAUR|A Epic Drama of a...|        2006|          1|                NULL|              6|       0.99|    86|           20.99|    PG|2022-09-10 16:46:...|"{""Deleted Scenes""|""Behind the Scen...|
|      2|  ACE GOLDFINGER|A Astounding Epis...|        2006|          1|                NULL|              3|       4.99|    48|        

In [7]:
film.write.mode("overwrite").parquet("film_parquet")
category.write.mode("overwrite").parquet("category_parquet")
film_category.write.mode("overwrite").parquet("film_category_parquet")
actor.write.mode("overwrite").parquet("actor_parquet")
film_actor.write.mode("overwrite").parquet("film_actor_parquet")
inventory.write.mode("overwrite").parquet("inventory_parquet")
rental.write.mode("overwrite").parquet("rental_parquet")
payment.write.mode("overwrite").parquet("payment_parquet")
customer.write.mode("overwrite").parquet("customer_parquet")
address.write.mode("overwrite").parquet("address_parquet")
city.write.mode("overwrite").parquet("city_parquet")

In [8]:
film = spark.read.parquet("film_parquet")
category = spark.read.parquet("category_parquet")
film_category = spark.read.parquet("film_category_parquet")
actor = spark.read.parquet("actor_parquet")
film_actor = spark.read.parquet("film_actor_parquet")
inventory = spark.read.parquet("inventory_parquet")
rental = spark.read.parquet("rental_parquet")
payment = spark.read.parquet("payment_parquet")
customer = spark.read.parquet("customer_parquet")
address = spark.read.parquet("address_parquet")
city = spark.read.parquet("city_parquet")

Output the number of movies in each category, sorted in descending order.

In [9]:
film_with_category = film_category.join(
    category,
    on = "category_id",
    how = "inner"
)

film_with_category.show(5)

+-----------+-------+-------------------+-----------+-------------------+
|category_id|film_id|        last_update|       name|        last_update|
+-----------+-------+-------------------+-----------+-------------------+
|          6|      1|2022-02-15 10:07:09|Documentary|2022-02-15 09:46:27|
|         11|      2|2022-02-15 10:07:09|     Horror|2022-02-15 09:46:27|
|          6|      3|2022-02-15 10:07:09|Documentary|2022-02-15 09:46:27|
|         11|      4|2022-02-15 10:07:09|     Horror|2022-02-15 09:46:27|
|          8|      5|2022-02-15 10:07:09|     Family|2022-02-15 09:46:27|
+-----------+-------+-------------------+-----------+-------------------+
only showing top 5 rows


In [10]:
movies_per_category = film_with_category.groupBy("name").agg(count("film_id").alias("movie_count"))
result = movies_per_category.orderBy(desc("movie_count"))
result.show()

+-----------+-----------+
|       name|movie_count|
+-----------+-----------+
|     Sports|         74|
|    Foreign|         73|
|     Family|         69|
|Documentary|         68|
|  Animation|         66|
|     Action|         64|
|        New|         63|
|      Drama|         62|
|      Games|         61|
|     Sci-Fi|         61|
|   Children|         60|
|     Comedy|         58|
|     Travel|         57|
|   Classics|         57|
|     Horror|         56|
|      Music|         51|
+-----------+-----------+



Output the 10 actors whose movies rented the most, sorted in descending order.

In [11]:
actor_rentals = actor \
.join(film_actor, on="actor_id", how = 'inner') \
.join(inventory, on = "film_id", how="inner") \
.join(rental, on="inventory_id", how= "inner")

actor_rentals.show(5)

+------------+-------+--------+----------+---------+-------------------+-------------------+--------+-------------------+---------+-------------------+-----------+-------------------+--------+-------------------+
|inventory_id|film_id|actor_id|first_name|last_name|        last_update|        last_update|store_id|        last_update|rental_id|        rental_date|customer_id|        return_date|staff_id|        last_update|
+------------+-------+--------+----------+---------+-------------------+-------------------+--------+-------------------+---------+-------------------+-----------+-------------------+--------+-------------------+
|           8|      1|       1|  PENELOPE|  GUINESS|2022-02-15 09:34:33|2022-02-15 10:05:03|       2|2022-02-15 10:09:17|    12651|2022-08-18 17:36:16|         34|2022-08-22 21:01:16|       1|2022-02-16 02:30:53|
|           8|      1|       1|  PENELOPE|  GUINESS|2022-02-15 09:34:33|2022-02-15 10:05:03|       2|2022-02-15 10:09:17|    10141|2022-07-31 21:08:

In [12]:
top_actors = actor_rentals \
  .groupBy("actor_id", "first_name", "last_name") \
  .agg(count("rental_id").alias("rental_count")) \
  .orderBy(desc('rental_count')) \
  .limit(10)

top_actors.show()

+--------+----------+-----------+------------+
|actor_id|first_name|  last_name|rental_count|
+--------+----------+-----------+------------+
|     107|      GINA|  DEGENERES|         753|
|     181|   MATTHEW|     CARREY|         678|
|     198|      MARY|     KEITEL|         674|
|     144|    ANGELA|WITHERSPOON|         654|
|     102|    WALTER|       TORN|         640|
|      60|     HENRY|      BERRY|         612|
|     150|     JAYNE|      NOLTE|         611|
|      37|       VAL|     BOLGER|         605|
|      23|    SANDRA|     KILMER|         604|
|      90|      SEAN|    GUINESS|         599|
+--------+----------+-----------+------------+



Output the category of movies on which the most money was spent.

In [13]:
category_revenue = payment \
    .join(rental, on="rental_id", how="inner") \
    .join(inventory, on="inventory_id", how="inner") \
    .join(film_category, on="film_id", how="inner") \
    .join(category, on="category_id", how="inner")


revenue_per_category = category_revenue \
  .groupBy('name') \
  .agg(spark_sum("amount").alias("total_revenue")) \
  .orderBy(desc("total_revenue"))

In [14]:
top_category = revenue_per_category.limit(1)

top_category.show()

+------+-----------------+
|  name|    total_revenue|
+------+-----------------+
|Sports|5314.209999999847|
+------+-----------------+



Output the names of movies that are not in the inventory.

In [15]:
movies_not_in_inventory = film.join(
    inventory,
    on="film_id",
    how="left_anti"
)

movies_not_in_inventory.select("title").show()

+--------------------+
|               title|
+--------------------+
|      ALICE FANTASIA|
|         APOLLO TEEN|
|      ARGONAUTS TOWN|
|       ARK RIDGEMONT|
|ARSENIC INDEPENDENCE|
|   BOONDOCK BALLROOM|
|       BUTCH PANTHER|
|       CATCH AMISTAD|
| CHINATOWN GLADIATOR|
|      CHOCOLATE DUCK|
|COMMANDMENTS EXPRESS|
|    CROSSING DIVORCE|
|     CROWDS TELEMARK|
|    CRYSTAL BREAKING|
|          DAZED PUNK|
|DELIVERANCE MULHO...|
|   FIREHOUSE VIETNAM|
|       FLOATS GARDEN|
|FRANKENSTEIN STRA...|
|  GLADIATOR WESTWARD|
+--------------------+
only showing top 20 rows


Output the top 3 actors who have appeared most in movies in the “Children” category. If several actors have the same number of movies, output all of them.

In [18]:
actors_with_categories = actor \
  .join(film_actor, on = "actor_id", how = "inner") \
  .join(film_category, on = "film_id", how = "inner") \
  .join(category, on = "category_id", how = "inner")

actors_with_categories.show(5)

+-----------+-------+--------+----------+---------+-------------------+-------------------+-------------------+-----------+-------------------+
|category_id|film_id|actor_id|first_name|last_name|        last_update|        last_update|        last_update|       name|        last_update|
+-----------+-------+--------+----------+---------+-------------------+-------------------+-------------------+-----------+-------------------+
|          6|      1|       1|  PENELOPE|  GUINESS|2022-02-15 09:34:33|2022-02-15 10:05:03|2022-02-15 10:07:09|Documentary|2022-02-15 09:46:27|
|          2|     23|       1|  PENELOPE|  GUINESS|2022-02-15 09:34:33|2022-02-15 10:05:03|2022-02-15 10:07:09|  Animation|2022-02-15 09:46:27|
|         13|     25|       1|  PENELOPE|  GUINESS|2022-02-15 09:34:33|2022-02-15 10:05:03|2022-02-15 10:07:09|        New|2022-02-15 09:46:27|
|         10|    106|       1|  PENELOPE|  GUINESS|2022-02-15 09:34:33|2022-02-15 10:05:03|2022-02-15 10:07:09|      Games|2022-02-15 09

In [28]:
from pyspark.sql.functions import col, countDistinct, dense_rank
from pyspark.sql.window import Window

children_movies = actors_with_categories \
  .filter(col("name") == "Children")

actor_counts = children_movies \
  .groupBy("actor_id", "first_name", "last_name") \
  .agg(countDistinct("film_id").alias("films_count"))

window_spec = Window.orderBy(desc("films_count"))

ranked_actors = actor_counts \
  .withColumn("rank", dense_rank().over(window_spec))

top_actors = ranked_actors \
  .filter(col("rank") <= 3) \
  .orderBy(desc("films_count"))

top_actors.show()

+--------+----------+---------+-----------+----+
|actor_id|first_name|last_name|films_count|rank|
+--------+----------+---------+-----------+----+
|      17|     HELEN|   VOIGHT|          7|   1|
|     127|     KEVIN|  GARLAND|          5|   2|
|      80|     RALPH|     CRUZ|          5|   2|
|      66|      MARY|    TANDY|          5|   2|
|     140|    WHOOPI|     HURT|          5|   2|
|      81|  SCARLETT|    DAMON|          4|   3|
|     109| SYLVESTER|     DERN|          4|   3|
|      23|    SANDRA|   KILMER|          4|   3|
|     187|     RENEE|     BALL|          4|   3|
|      92|   KIRSTEN|   AKROYD|          4|   3|
|     173|      ALAN| DREYFUSS|          4|   3|
|     101|     SUSAN|    DAVIS|          4|   3|
|     150|     JAYNE|    NOLTE|          4|   3|
|      13|       UMA|     WOOD|          4|   3|
|     131|      JANE|  JACKMAN|          4|   3|
|      58| CHRISTIAN|   AKROYD|          4|   3|
|     142|      JADA|    RYDER|          4|   3|
|      93|     ELLEN

Output cities with the number of active and inactive customers (active - customer.active = 1). Sort by the number of inactive customers in descending order.

In [33]:
from pyspark.sql.functions import col, sum as spark_sum, when, desc

cities_with_customers = customer \
    .join(address, on="address_id", how="inner") \
    .join(city, on="city_id", how="inner")

city_stats = cities_with_customers \
    .groupBy("city") \
    .agg(
        spark_sum(when(col("active") == 1, 1).otherwise(0)).alias("active_count"),
        spark_sum(when(col("active") == 0, 1).otherwise(0)).alias("inactive_count")
    ) \
    .orderBy("inactive_count")

city_stats.show()

+------------------+------------+--------------+
|              city|active_count|inactive_count|
+------------------+------------+--------------+
|A Corua (La Corua)|           1|             0|
|          Fengshan|           1|             0|
|          Myingyan|           1|             0|
|          Chisinau|           1|             0|
|              Linz|           1|             0|
|           Udaipur|           1|             0|
|           El Alto|           1|             0|
|               Oyo|           1|             0|
|      Juiz de Fora|           1|             0|
|           Esfahan|           1|             0|
|            Monywa|           1|             0|
|       Sultanbeyli|           1|             0|
|    Dhule (Dhulia)|           1|             0|
|         Mit Ghamr|           1|             0|
|            Jining|           1|             0|
|          Salzburg|           1|             0|
|           Tanauan|           1|             0|
|          Sogamoso|

Output the category of movies that have the highest number of total rental hours in the cities (customer.address_id in this city), and that start with the letter “a”. Do the same for cities with a “-” symbol.

In [34]:
from pyspark.sql.functions import col, sum as spark_sum, desc, lower
from pyspark.sql.functions import unix_timestamp

full_data = rental \
    .join(inventory, "inventory_id") \
    .join(film_category, "film_id") \
    .join(category, "category_id") \
    .join(customer, "customer_id") \
    .join(address, "address_id") \
    .join(city, "city_id")

full_data = full_data.filter(col("return_date").isNotNull())

full_data = full_data.withColumn(
    "rental_hours",
    (unix_timestamp("return_date") - unix_timestamp("rental_date")) / 3600
)

In [35]:
cities_a = full_data.filter(lower(col("city")).startswith("a"))
result_a = cities_a \
    .groupBy("name") \
    .agg(spark_sum("rental_hours").alias("total_hours")) \
    .orderBy(desc("total_hours")) \
    .limit(1)

result_a.show()

+------+------------------+
|  name|       total_hours|
+------+------------------+
|Sports|12360.349999999999|
+------+------------------+



In [36]:
cities_dash = full_data.filter(col("city").contains("-"))

result_dash = cities_dash \
    .groupBy("name") \
    .agg(spark_sum("rental_hours").alias("total_hours")) \
    .orderBy(desc("total_hours")) \
    .limit(1)

result_dash.show()

+-------+-----------+
|   name|total_hours|
+-------+-----------+
|Foreign|    6472.15|
+-------+-----------+



In [37]:
spark.stop()